# LangChain: RAG & Q&A

## Outline
* RAG concept
* Build a knowledge base (loader → split → embed → store)
* 2-Step RAG with LCEL
* Agentic RAG with `create_agent`
* Comparison of the two approaches


In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

_ = load_dotenv(find_dotenv())

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0, api_key=api_key, base_url=base_url)
ollama = init_chat_model("llama3.1:8b ", model_provider="ollama", temperature=0)

## 1. RAG Concept

**RAG (Retrieval-Augmented Generation)** solves two limitations of LLMs:
- **Limited context**: it cannot read an entire corpus at once
- **Static knowledge**: training data is frozen at a specific point in time

Solution: at query time, fetch relevant information from external sources and provide it to the LLM.

```
User Query → Retriever → Relevant Docs → LLM → Answer
```


## 2. Build a Knowledge Base

In [11]:
%pip uninstall torchcodec

^C
Note: you may need to restart the kernel to use updated packages.


In [13]:
# pip install langchain-openai langchain-community faiss-cpu

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
#from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain.text_splitter import RecursiveCharacterTextSplitter


# ── Step 1: Load ──
# loader = CSVLoader(file_path="OutdoorClothingCatalog_1000.csv")
# docs = loader.load()

# For the demo, we create a few documents manually
from langchain_core.documents import Document

docs = [
    Document(page_content="The sun-protection shirt has a UPF 50+ rating and blocks 98 percent of ultraviolet rays. Machine washable.", metadata={"source": "catalog", "id": 1}),
    Document(page_content="Our swimsuits are very comfortable and dry quickly. Available in 6 colors.", metadata={"source": "catalog", "id": 2}),
    Document(page_content="The hiking boots are waterproof and provide excellent ankle support on hiking trails.", metadata={"source": "catalog", "id": 3}),
    Document(page_content="The sun-protection hat blocks ultraviolet rays and has a wide brim. Lightweight design.", metadata={"source": "catalog", "id": 4}),
    Document(page_content="The fleece jacket retains warmth in cold weather. Machine washable and easy to pack.", metadata={"source": "catalog", "id": 5}),
]
print(f"Loaded {len(docs)} documents")
print(f"Sample: {docs[-1].page_content[:100]}")



Loaded 5 documents
Sample: The fleece jacket retains warmth in cold weather. Machine washable and easy to pack.


In [20]:
# ── Step 2: Split ──
splitter = RecursiveCharacterTextSplitter(
    #chunk_size=50,
    chunk_size=200,
    chunk_overlap=20,
)
splits = splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks")


Split into 5 chunks


In [21]:
splits

[Document(metadata={'source': 'catalog', 'id': 1}, page_content='The sun-protection shirt has a UPF 50+ rating and blocks 98 percent of ultraviolet rays. Machine washable.'),
 Document(metadata={'source': 'catalog', 'id': 2}, page_content='Our swimsuits are very comfortable and dry quickly. Available in 6 colors.'),
 Document(metadata={'source': 'catalog', 'id': 3}, page_content='The hiking boots are waterproof and provide excellent ankle support on hiking trails.'),
 Document(metadata={'source': 'catalog', 'id': 4}, page_content='The sun-protection hat blocks ultraviolet rays and has a wide brim. Lightweight design.'),
 Document(metadata={'source': 'catalog', 'id': 5}, page_content='The fleece jacket retains warmth in cold weather. Machine washable and easy to pack.')]

In [24]:
# ── Step 3: Embed & Store ──
from sklearn.metrics.pairwise import cosine_similarity

embeddings = OpenAIEmbeddings(model="text-embedding-3-large", api_key=api_key, base_url=base_url)

# Test embedding
sample_embed1 = embeddings.embed_query("King")
sample_embed2 = embeddings.embed_query("Qween")
all_token = [sample_embed1, sample_embed2]

print(f"Embedding dimension: {len(sample_embed1)}")
print(f"First 5 values: {sample_embed1[:5]}")

print(cosine_similarity(all_token))

Embedding dimension: 3072
First 5 values: [0.01122283935546875, 0.01309967041015625, -0.0130462646484375, 0.0033111572265625, -0.007602691650390625]
[[1.         0.43243015]
 [0.43243015 1.        ]]


In [28]:
# Build the vector store
vectorstore = FAISS.from_documents(splits, embeddings)
print("Vector store created!")

# Test similarity search
query = "shirts with sun protection"
results = vectorstore.similarity_search(query, k=2)
print(f"\nTop {len(results)} results for '{query}':")
for i, doc in enumerate(results):
    print(f"  {i+1}. {doc.page_content[:200]}")


Vector store created!

Top 2 results for 'shirts with sun protection':
  1. The sun-protection shirt has a UPF 50+ rating and blocks 98 percent of ultraviolet rays. Machine washable.
  2. The sun-protection hat blocks ultraviolet rays and has a wide brim. Lightweight design.


In [ ]:
# Change the method to retrieve documents along with similarity scores
results = vectorstore.similarity_search_with_relevance_scores(query, k=2)       # Just score added
print(f"\nTop {len(results)} results for '{query}':")

# Unpack the (doc, score) pair in the loop
for i, (doc, score) in enumerate(results):
    percentage = score * 100  # Convert the score to a percentage
    print(f"  {i+1}. {doc.page_content[:200]} (Similarity: {percentage:.2f}%)")


Top 2 results for 'shirts with sun protection':
  1. The sun-protection shirt has a UPF 50+ rating and blocks 98 percent of ultraviolet rays. Machine washable. (Similarity: 57.40%)
  2. The sun-protection hat blocks ultraviolet rays and has a wide brim. Lightweight design. (Similarity: 37.00%)


### Full example

In [34]:
import numpy as np
import faiss
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import os
from dotenv import load_dotenv

load_dotenv()

# ============================================================
# 1. Create Documents
# ============================================================

docs = [
    Document(page_content="The sun-protection shirt has a UPF 50+ rating and blocks 98 percent of ultraviolet rays. Machine washable.", metadata={"id": 1}),
    Document(page_content="Our swimsuits are very comfortable and dry quickly. Available in 6 colors.", metadata={"id": 2}),
    Document(page_content="The hiking boots are waterproof and provide excellent ankle support on hiking trails.", metadata={"id": 3}),
    Document(page_content="The sun-protection hat blocks ultraviolet rays and has a wide brim. Lightweight design.", metadata={"id": 4}),
    Document(page_content="The fleece jacket retains warmth in cold weather. Machine washable and easy to pack.", metadata={"id": 5}),
]

# ============================================================
# 2. Split Documents
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    #separator="."
)

splits = text_splitter.split_documents(docs)
print(f"📄 Created {len(splits)} chunks\n")

# Show chunks
for i, chunk in enumerate(splits):
    print(f"Chunk {i+1}: {chunk.page_content[:100]}...")

# ============================================================
# 3. Create Vector Store (FAISS)
# ============================================================

vectorstore = FAISS.from_documents(splits, embeddings)
print("\n✅ Vector store created!")

# ============================================================
# 4. Inspect the FAISS Index
# ============================================================

# FAISS stores vectors internally - let's see what's inside
print("\n" + "="*60)
print("🔍 FAISS Index Information")
print("="*60)

# Access the underlying FAISS index
faiss_index = vectorstore.index
print(f"Index type: {type(faiss_index)}")
print(f"Number of vectors: {faiss_index.ntotal}")
print(f"Vector dimension: {faiss_index.d}")

# ============================================================
# 5. Search Process Visualization
# ============================================================

def search_with_visualization(query: str, k: int = 2):
    """
    Perform a search and show the process step-by-step.
    """
    print("\n" + "="*60)
    print(f"🔍 SEARCH: '{query}'")
    print("="*60)
    
    # Step 1: Generate query embedding
    print("\n📊 Step 1: Generating query embedding...")
    query_embedding = embeddings.embed_query(query)
    print(f"   Query vector dimension: {len(query_embedding)}")
    print(f"   First 5 values: {query_embedding[:5]}...")
    
    # Step 2: Search the FAISS index
    print(f"\n🔎 Step 2: Searching FAISS index (k={k})...")
    results = vectorstore.similarity_search_with_score(query, k=k)
    
    # Step 3: Show results
    print(f"\n📄 Step 3: Found {len(results)} similar chunks:")
    for i, (doc, score) in enumerate(results):
        print(f"\n   Result {i+1}:")
        print(f"   ──────────────────────────────")
        print(f"   Content: {doc.page_content[:150]}...")
        print(f"   Distance (L2 score): {score:.4f}")  # Lower = more similar
        print(f"   Metadata: {doc.metadata}")
    
    return results

# ============================================================
# 6. Test Different Queries
# ============================================================

# Query 1: Direct match
search_with_visualization("shirts with sun protection", k=2)

# Query 2: Partial match
search_with_visualization("waterproof boots", k=2)

# Query 3: General query
search_with_visualization("warm clothing", k=2)

# ============================================================
# 7. Saving and Loading FAISS Index
# ============================================================

print("\n" + "="*60)
print("💾 SAVING AND LOADING")
print("="*60)

# Save to disk
vectorstore.save_local("faiss_index")
print("✅ Saved to 'faiss_index' folder")

# Load from disk (requires embeddings object)
loaded_vectorstore = FAISS.load_local(
    "faiss_index", 
    embeddings,
    allow_dangerous_deserialization=True
)
print("✅ Loaded from disk")

# Verify it works
test_results = loaded_vectorstore.similarity_search("sun protection", k=1)
print(f"✅ Loaded index works: {test_results[0].page_content[:80]}...")

# ============================================================
# 8. Comparison: Different Search Methods
# ============================================================

print("\n" + "="*60)
print("📊 COMPARING SEARCH METHODS")
print("="*60)

query = "waterproof boots"

# Method 1: similarity_search (returns documents)
results1 = vectorstore.similarity_search(query, k=2)
print("\n1️⃣ similarity_search() - returns Documents only:")
for i, doc in enumerate(results1):
    print(f"   {i+1}. {doc.page_content[:80]}...")

# Method 2: similarity_search_with_score (returns Documents + scores)
results2 = vectorstore.similarity_search_with_score(query, k=2)
print("\n2️⃣ similarity_search_with_score() - returns Documents + scores:")
for i, (doc, score) in enumerate(results2):
    print(f"   {i+1}. Score: {score:.4f} | {doc.page_content[:80]}...")

# Method 3: as_retriever (for RAG chains)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
results3 = retriever.get_relevant_documents(query)
print("\n3️⃣ as_retriever() - for use in RAG chains:")
for i, doc in enumerate(results3):
    print(f"   {i+1}. {doc.page_content[:80]}...")

📄 Created 5 chunks

Chunk 1: The sun-protection shirt has a UPF 50+ rating and blocks 98 percent of ultraviolet rays. Machine was...
Chunk 2: Our swimsuits are very comfortable and dry quickly. Available in 6 colors....
Chunk 3: The hiking boots are waterproof and provide excellent ankle support on hiking trails....
Chunk 4: The sun-protection hat blocks ultraviolet rays and has a wide brim. Lightweight design....
Chunk 5: The fleece jacket retains warmth in cold weather. Machine washable and easy to pack....

✅ Vector store created!

🔍 FAISS Index Information
Index type: <class 'faiss.swigfaiss.IndexFlatL2'>
Number of vectors: 5
Vector dimension: 3072

🔍 SEARCH: 'shirts with sun protection'

📊 Step 1: Generating query embedding...
   Query vector dimension: 3072
   First 5 values: [-0.01258087158203125, -0.03802490234375, -0.006259918212890625, 0.02362060546875, 0.02020263671875]...

🔎 Step 2: Searching FAISS index (k=2)...

📄 Step 3: Found 2 similar chunks:

   Result 1:
   ─────────

AttributeError: 'VectorStoreRetriever' object has no attribute 'get_relevant_documents'

## 3. Two-Step RAG with LCEL

In [36]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0, api_key=api_key, base_url=base_url)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the question based only on the following context.


Context:
{context}"""),
("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)



# 2-Step RAG Chain
rag_chain = ( rag_prompt
            | llm
            | StrOutputParser()
)

query = "Please list all shirts that provide sun protection"
retriever_results = format_docs(vectorstore.similarity_search(query, k=2))

response = rag_chain.invoke({"context":retriever_results , "question": query})
print(response)


The context mentions a sun-protection shirt with a UPF 50+ rating that blocks 98 percent of ultraviolet rays. This is the only shirt specifically mentioned that provides sun protection.


In [38]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0, api_key=api_key, base_url=base_url)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the question based only on the following context.
If you don't know the answer, say "I don't have that information."

Context:
{context}"""),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)



# 2-Step RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test
response = rag_chain.invoke("Please list all shirts that provide sun protection")
print(response)


The context mentions a sun-protection shirt with a UPF 50+ rating that blocks 98 percent of ultraviolet rays. This is the only shirt providing sun protection mentioned.


In [ ]:
# Streaming with RAG
print("Streaming RAG response:")
for chunk in rag_chain.stream("What clothing do you have for cold weather?"):
    print(chunk, end="", flush=True)
print()


## 4. Agentic RAG with `create_agent`

In [41]:
from langchain.agents import create_agent
from langchain.tools import tool

# RAG as a tool
@tool
def search_catalog(query: str) -> str:
    """Search the product catalog for relevant items.
    Use this when the user asks about products, clothing, or outdoor gear."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant products found."
    return "\n".join([f"- {doc.page_content}" for doc in docs])

# Build the RAG agent
rag_agent = create_agent(
    #model=ollama,
    model=llm,
    tools=[search_catalog],
    system_prompt="""You are a helpful outdoor clothing store assistant.
                    Use the search_catalog tool to find relevant products before answering questions.
                    Always base your answers on the search results."""
)


In [42]:
response = rag_agent.invoke({
    "messages": [{"role": "user", "content": "What products do you have for sun protection?"}]
})
print(response["messages"][-1].content)


Here are some products for sun protection that we have:

1. **Sun-Protection Hat**: This hat blocks ultraviolet rays and features a wide brim. It has a lightweight design for comfort.

2. **Sun-Protection Shirt**: This shirt has a UPF 50+ rating, blocking 98 percent of ultraviolet rays. It's also machine washable for easy care.

3. **Swimsuits**: Our swimsuits are designed for comfort and dry quickly. They are available in six colors.

Let me know if you need more information on any of these products!


In [ ]:
# Streaming agent
print("Agent response:")
for chunk in rag_agent.stream(
    {"messages": [{"role": "user", "content": "I need hiking boots. What do you recommend?"}]},
    stream_mode="values"
):
    latest = chunk["messages"][-1]
    if hasattr(latest, "content") and latest.content:
        print(latest.content)


In [43]:
response = rag_agent.invoke({
    "messages": [{"role": "user", "content": "What is the capital of Iran?"}]
})
print(response["messages"][-1].content)


The capital of Iran is Tehran. If you have any questions about outdoor clothing or gear, feel free to ask!


## 5. Comparison: 2-Step RAG vs Agentic RAG

| | 2-Step RAG | Agentic RAG |
|---|---|---|
| **Method** | Always retrieve, then generate | The agent decides when to retrieve |
| **Speed** | Faster, predictable | Slower, variable |
| **Flexibility** | Low — always one retrieval | High — can search multiple times |
| **Control** | High | Lower |
| **Best suited for** | FAQ, documentation bot | research assistant, multi-source |


## Summary — Required Installation

```bash
pip install langchain langchain-openai langchain-community faiss-cpu
```

**Main RAG Pipeline:**
1. **Load** — `CSVLoader`, `PyPDFLoader`, `WebBaseLoader`
2. **Split** — `RecursiveCharacterTextSplitter`
3. **Embed** — `OpenAIEmbeddings`
4. **Store** — `FAISS`, `Chroma`, `PineconeVectorStore`
5. **Retrieve** — `vectorstore.as_retriever()`
6. **Generate** — LCEL chain or `create_agent`
